# Actual chat transport and attached tokenizer probe41
CPU only. Same frozen v10 agent, actual organizer compiler, ADK Runner, LiteLLM
and OpenAI serialization. Mock HTTP transport provides three declared synthetic
responses; real load_skill and get_status run. No HTTP socket, model weights,
Gemma inference, official task, grader or Stage1 loader.
The attached required31B QAT revision2 tokenizer renders these captured requests.
The installed vLLM pure argument postprocessor is extracted without GPU imports;
this does not execute the full server parser. A preserve_thinking override is a
separate diagnostic observation and does not change any scoring candidate.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, base64, zipfile, io, time, inspect
IS_KAGGLE = Path('/kaggle/working').is_dir()
RUN_DECLARED_EXPERIMENT = True
MODE = 'cpu_controls'
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
if not IS_KAGGLE:
    print('Local definitions only: no installation, task read, inference or grader call')


In [ ]:
"""Export only artifacts produced inside a run, including per-trial checkpoints."""
import base64
import hashlib
import io
import json
import os
from pathlib import Path
import tempfile
import zipfile


def export_run(run_root: Path, *, emit: bool = True) -> dict:
    run_root = run_root.resolve(strict=True)
    artifacts, manifest = [], []
    for artifact in sorted(run_root.rglob('*')):
        resolved = artifact.resolve()
        if artifact.is_symlink() or not resolved.is_relative_to(run_root):
            raise ValueError('Run artifact is a link or escaped its directory')
        name = artifact.relative_to(run_root).as_posix()
        if name == 'evidence_bundle.zip' or name.startswith('evidence_checkpoints/'):
            continue
        if not artifact.is_file():
            continue
        if name == 'artifact_manifest.json':
            raise ValueError('artifact_manifest.json is reserved for the exported manifest')
        # Read once. A live log may grow, but its archived bytes and hash must agree.
        payload = artifact.read_bytes()
        artifacts.append((name, payload))
        manifest.append({'path': name, 'bytes': len(payload),
                         'sha256': hashlib.sha256(payload).hexdigest()})
    if sum(item['bytes'] for item in manifest) > 32 * 1024 * 1024:
        raise ValueError('Use Kaggle output download for this large evidence bundle')
    bundle = io.BytesIO()
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
        for name, payload in artifacts:
            archive.writestr(name, payload)
        archive.writestr('artifact_manifest.json', json.dumps(manifest, indent=2) + '\n')
    bundle_bytes = bundle.getvalue()
    digest = hashlib.sha256(bundle_bytes).hexdigest()
    checkpoints = run_root / 'evidence_checkpoints'
    checkpoints.mkdir(exist_ok=True)
    checkpoint = checkpoints / (digest + '.zip')
    if checkpoint.exists():
        if checkpoint.read_bytes() != bundle_bytes:
            raise ValueError('Existing checkpoint differs from its content hash')
    else:
        with checkpoint.open('xb') as stream:
            stream.write(bundle_bytes)
            stream.flush()
            os.fsync(stream.fileno())
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=run_root, prefix='.evidence_bundle.',
                                         suffix='.tmp', delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(bundle_bytes)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, run_root / 'evidence_bundle.zip')
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()
    report = {'sha256': digest, 'bytes': len(bundle_bytes), 'file_count': len(manifest),
              'checkpoint_path': checkpoint.relative_to(run_root).as_posix(),
              'snapshot_is_atomic_across_all_files': False}
    if emit:
        print('NEURAL_SWE_BUNDLE_SHA256=' + report['sha256'], flush=True)
        print('NEURAL_SWE_BUNDLE_BASE64=' + base64.b64encode(bundle_bytes).decode('ascii'), flush=True)
    return report


In [ ]:
PLAN = json.loads('{"expected_core_packages": {"swegemma": "0.2.7", "adk-submission": "0.2.12", "adk-eval-core": "0.1.0"}, "expected_verify_task_sha256": "98600fd6066a4dbd472490cb194b839a3a633afea9b99256cc081ea77beac0d3", "expected_organizer_python_sources": [{"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/__init__.py", "bytes": 3857, "sha256": "355aaa3a1cf2b4b3bb5b5b6b3e72acc3481c657918411b4f18bde76415284639"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/__main__.py", "bytes": 145, "sha256": "5cd5c92ee1a230e24c17d693c3093edf0e3e78b96c08815351ba82432a1aa4f4"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/budget.py", "bytes": 1305, "sha256": "60bd0f73e0fcede68a3024a5e37836b0bcb0d133952044d7dc3e5a4102f2c2f9"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/cli.py", "bytes": 5221, "sha256": "7197c5fd6b141d7df805dd350130944c84b0b1fc14ed5eeef3f52b2e26d5e30d"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/config.py", "bytes": 7440, "sha256": "f035916218f895796c4d39f1f9ec25284152a6df57cf6de1fea4931165d2349f"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/context.py", "bytes": 23710, "sha256": "97f06280c50df3132d1af7ef6126884b66b48887a1cf94b293af01e5907dc2a7"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/deduplication.py", "bytes": 49745, "sha256": "f8353a8da887fc45823b9174f4578cd816b341d52c3a074964c1c2310af1573e"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/difficulty.py", "bytes": 16760, "sha256": "0b97e060fa6a3b437024a45c42cb131526d5a48b74dd858e4fc0259301f7a6ba"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/display.py", "bytes": 3976, "sha256": "d2dfff4ebe0ff655a240b5c1f073e85702869f618c0aa5c95cc6b54b20be47c8"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/edit.py", "bytes": 2573, "sha256": "d34acdff4530ed211325117395cf7fa9d085f7ee84250d02acf9c8533657a956"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/evaluate.py", "bytes": 25197, "sha256": "96a46f49267ffcf861f7f90b98efbe828d03f27553ceefce745819028589de1e"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/graph/__init__.py", "bytes": 1372, "sha256": "9d9c6974861db6d3bea45cf2dee60ce1ba9073293138ad5157ad696ca23d4f7a"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/graph/embedding_utils.py", "bytes": 14515, "sha256": "eb1c777ea521416f56e60ad258dc14c412c3afca1687e7244a01d59657b641c9"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/graph/graph_utils.py", "bytes": 15651, "sha256": "e48d9f0cabe02e9ce8b1a0ab437de68a90e0a740b61a3f54ce5a95ab1f05c1cc"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/graph/retrieval_utils.py", "bytes": 32100, "sha256": "5d18a87de66af0c773de7d4601f37c74d5fb575c9fc73763cc803a79fa74d84d"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/harness/__init__.py", "bytes": 1079, "sha256": "ca5053d64fc35f7d457974879656e63fa7980a7b36d37a28bbe7391f1fa90fe5"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/harness/agent_runner.py", "bytes": 35490, "sha256": "dff7d2505340633c3fe50ba7073f2609fb2b41becd69c356d9af25cc4c1f98dd"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/harness/container_setup.py", "bytes": 40661, "sha256": "3192acdce35f2d0bd0b191186851b46c251390af16b7f139a1640e16fb1d862c"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/harness/sample_verification.py", "bytes": 20399, "sha256": "0bbd4322534442420a4f782ffb76a5467d6a9d9bd7c32fe212269f304e05fa41"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/harness/verification.py", "bytes": 22102, "sha256": "5602f33acb49062de8e99073199745679d86445297be9dd075ccec60c41b60dd"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/models/__init__.py", "bytes": 793, "sha256": "34ecd27654b031cd3ede1cead44ed0e8f72bbfe6ffea24c22bfe66d493edbd97"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/models/discovery.py", "bytes": 5681, "sha256": "70ff2d1719f71cd4c0d747fecfcf5763c1fbbb983684cc2d68b6771e3a3997ca"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/models/registry.py", "bytes": 7374, "sha256": "69e62c69d461a11f1655285652872005aee11aa549beaec331742dc742593ca3"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/models/task.py", "bytes": 7693, "sha256": "0b9d705ce759645bd1ee59bb28662a25a951990c024bfeb1e00a41f08350812a"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/results.py", "bytes": 5200, "sha256": "4705de95a05ae5f01f743cbb97046fb0f47f1407f4cd1db249489b2253a7f72f"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/sandbox/__init__.py", "bytes": 829, "sha256": "ba3725efee8f6d0fba2321851d66d4497ae3cda5863edc0d4a8ef232f1fdb775"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/sandbox/base.py", "bytes": 10113, "sha256": "aab491aabc84105963bead16dab4d3d5745e31727ba248fd5a568d8f4e0c7dc1"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/sandbox/docker.py", "bytes": 34531, "sha256": "cc10c287acf7c9326f622e91b98207bb9cdcf4fadf5d4aa5f5658e854051d09d"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/sandbox/subprocess.py", "bytes": 21429, "sha256": "a26e9265d93f0387dd0a820ea451d1c530884bebca41583738f3ab2cc31df1f2"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/submission.py", "bytes": 3577, "sha256": "d387b402223300d06eb6f4f960faf82e997098050f5317961c64e6b95365974a"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/tools/__init__.py", "bytes": 1632, "sha256": "e19d1e81d90976fda93f346104d7f34f74018747ac81838a363de2e357358c31"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/tools/base.py", "bytes": 3768, "sha256": "535a0e6d121549b41906685d0e440e8709c4b8b434bc31dcc82fa7740ec66281"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/tools/execution.py", "bytes": 6055, "sha256": "f2ce1cefb41edddaeffeacbff910a54bcd0174b548efa9c6c283320e5a509912"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/tools/graph.py", "bytes": 16411, "sha256": "7f102bf348fe799b89cd8d614afea35c5a21d8141a29859582363086ce2b1173"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/tools/workspace.py", "bytes": 12316, "sha256": "d38ba2f68636c5e33c9f607bd9e5401dbf81d77125b966ef87c799a1615b05c6"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/warnings.py", "bytes": 1084, "sha256": "f24d5cf69977bd88a5bb82639767ba41c12fed3bd8692969cb7c80051fbb1945"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/__init__.py", "bytes": 4369, "sha256": "ebe048f521fd5bcf46dad132f3e3dea2506df79bf471d560a70ce5b343ca387a"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/builders/__init__.py", "bytes": 637, "sha256": "ec6273c54cfb08f22777146c1636be4f13d4d00a67a17d9ae7a19e3a8f18f6c1"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/builders/llm.py", "bytes": 3658, "sha256": "97c27324ce60ac6dc466941bcc19f5482f7693af9fc02fb760a40526c84c10b5"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/builders/sub_agents.py", "bytes": 2801, "sha256": "ba6162832b1ca4e023e2848c6391242c18cbe9dadcfd560a6ebef81108c35b5a"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/builders/workflows.py", "bytes": 5565, "sha256": "d8d7ccf9a51d57daec58c7231e2a62621a3ecca269e71ee64c57062cadadf980"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/compiler.py", "bytes": 10492, "sha256": "39cc4884699faf147ca1b326664e888c1861f750fe36525b840b5f812d593eed"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/context.py", "bytes": 7403, "sha256": "3c77519415b1dd2cc97b42bcbe29d32dd85f8188ca7eba5b5fb8636bce92c908"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/discovery.py", "bytes": 31163, "sha256": "ca19fc75fb3baa42e85438b41248a06c35987cbf26ef652159b302752d0f191c"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/errors.py", "bytes": 13999, "sha256": "7e121e7c6581288788500cd1a8deceb003a74ec150d5477c37f794971a5f4818"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/limits.py", "bytes": 18965, "sha256": "afde5056f2df1fd77b53e0495712d03f60b3e246d4393617dd800d970f7482e2"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/paths.py", "bytes": 9379, "sha256": "7c4b7050141bdb05a9e6849be11f6440f2a18082a364623226c4343649b2f59f"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/registry.py", "bytes": 11173, "sha256": "c17f9770c2f3663309dd61807d2ae4e7d75bbd06b7c674847f655d218e82cf7a"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/__init__.py", "bytes": 800, "sha256": "b88fe281c2b927a046e6485c7e0966b1808a5c8d0a0849cd85824ffb04ae53be"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/callbacks.py", "bytes": 3643, "sha256": "332ce0732aa3eb1b555630c57683f51e59d5caee3f519371d376210e9ae33e29"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/generation.py", "bytes": 13605, "sha256": "ba8bf237261220d46843c8d71118faa387fc07f876f072756dff44d641dd1c14"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/models.py", "bytes": 3179, "sha256": "39070f6ae52ad8880750b51af373da8762850d3a8ecc65c6f11744afe4cd5066"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/tools.py", "bytes": 10593, "sha256": "ae44a5ca815a05edfd763b92283d0444c77755afbd827fc35bcb87eb2633259e"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/schema.py", "bytes": 17404, "sha256": "6554de60cfa208dcb0b095f4236271795b2955adf87f3be3c40dfa3d62521c5d"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/server.py", "bytes": 35447, "sha256": "d60b3fe34dd7f6e359b003db82b68434757c74b1030862bacda5cff6164f91eb"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/yaml_loader.py", "bytes": 12618, "sha256": "01d644ef7f5acc216506245fd7e115e7096641e62e825c6873c83e148b23c69c"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/__init__.py", "bytes": 4122, "sha256": "528e02f35eaef6fa733acc0a026c80cc191d28d4a4192a740f9b7bc8a88c9211"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/budget/__init__.py", "bytes": 254, "sha256": "4a709384174fb5be5aa22c9a1385ba830d82bf03fab1798280b350311e75588e"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/budget/budget.py", "bytes": 15780, "sha256": "16de25ae326f0f8440772672a4bf7198a4ebb5586c7c3f2b31048ca896f04ab6"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/editing/__init__.py", "bytes": 266, "sha256": "6e4f2c12837a366147d582b9ccfc4f6c3fd0d117e311767e292abe8b84642397"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/editing/edit.py", "bytes": 14676, "sha256": "a78a761d5078df92606c63b3adb61991a2878bcbb80204875d241175cb532a6f"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/errors.py", "bytes": 5691, "sha256": "2a8d545695a89c0abbc8984d771df6ae31155998b8e20cb91da02d0a5b900373"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/models/__init__.py", "bytes": 246, "sha256": "1a9e1673840b8be5bcbf93767a1d130c007abf350cfc2c263e389135bba223af"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/models/base.py", "bytes": 9724, "sha256": "bdedef267b1022ffac970c7bc18c2d0a53753ce1f120a3c355b91c9b7b666a0d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/plugins/__init__.py", "bytes": 227, "sha256": "dd53fad958a5719da6d6ab6869f59daa7c70060a257dedc77551873557b829f9"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/plugins/display_plugin.py", "bytes": 5648, "sha256": "7250793fa30985086c193fc55d5b81f4c4d115b44ad764e546a8cc1b0ac20d69"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/plugins/retry_plugin.py", "bytes": 7084, "sha256": "ba61cf201167aaf970c3131a4f24b464515509f3cacd5358de99c5ec0fb07af0"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/runner/__init__.py", "bytes": 293, "sha256": "2faaa7d8445299656396987f7b55876025b5dfaab793afa0c11027ab08984b01"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/runner/base.py", "bytes": 1198, "sha256": "fa97acbfdf8be95b9b247e525fb4fa682c8b8da156b405f5bc29f684eafb4d5f"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/runner/runner.py", "bytes": 3321, "sha256": "0c057f402eb42f0bdc4e84fd88d064287f3929284d8f345813df8dedeb7b9cc3"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/__init__.py", "bytes": 751, "sha256": "b9f13699e2bc09fa57815bb3c6bf07c4aefb9ab5b74e80ec46625f8421b10d5a"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/base.py", "bytes": 7694, "sha256": "da53e3bb0dfb7d2490d63195740d94560291c37001c4ebae4d12733cf4856007"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/containers.py", "bytes": 21219, "sha256": "c36a7077901aafcd8a16ae22441dfe40d8713e36bbc8534ab2bad1cd953bf769"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/docker_sandbox.py", "bytes": 7011, "sha256": "484b8864c2ff876f10b85f95e69e72b245e8ca5f6f30dc1020baec846212c899"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/subprocess_sandbox.py", "bytes": 11715, "sha256": "09dc213bd71f4ad33a2a403732271efbc78bd09db2d2999817d55f4e630ac6fc"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/__init__.py", "bytes": 408, "sha256": "d1b4ed94c3c5fa97541f593d62645cc73fe60587da6d2deb742c48ac00f01096"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/atif.py", "bytes": 5302, "sha256": "266324e80f6a4fc23aa3d809f677a6e950a7bc971f8e903ce3f55dc2c0cd8f35"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/atif_converter.py", "bytes": 7430, "sha256": "1d35490d305e1676d09c11a54301488cdecc6aceb03ee3469831d3f8a1cdd992"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/markdown.py", "bytes": 7160, "sha256": "e8e07b6a71926196e09a18f08f0cc3ff6da5820bff95fcc0534fdeab083fbb6d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/trace.py", "bytes": 23186, "sha256": "b426e17c03b420f38ee4e08c038678ff4ba66d07ea893638f4e22a9fba8a4bc5"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/__init__.py", "bytes": 1571, "sha256": "3f39053c2fd360dedca776c611df55b0a4382e910f3681b0f9dbb3f08b0b459d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/dashboard.py", "bytes": 18158, "sha256": "c56edca53fe06efb29151ef140b25ac99f795b08faf0bf0632abae18ce544eec"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/event_display.py", "bytes": 23833, "sha256": "4efbb0a55b5cd93cd0d8036c2d3f5f65b67072a0f5dc197e2e7b360a791f4a3e"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/__init__.py", "bytes": 1170, "sha256": "2502098e77ae11a186177c980cb30560e3cd90703026a0c3dda95e191bbce288"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/command.py", "bytes": 4046, "sha256": "50856b977eb86fd21f312a340e6d782101b934564a3527348e869294aa84d960"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/file.py", "bytes": 5709, "sha256": "5ba0b3a5e51925431545a0f2f408725d4e2e487b934f821c635fb364b10511bf"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/generic.py", "bytes": 3922, "sha256": "f458b8993b7158c3a5a73991c0adb3cb35509cd927a8ab3b14df4a515020b3be"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/models.py", "bytes": 5146, "sha256": "5445d681fa4f4dc9847cda9388af8c623fd7e1e73bce15f8c42de9e34a8f84ea"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/python.py", "bytes": 2119, "sha256": "3249c42b0b2d73a447913d07076623f3dbd7401b63726d9e3a3a9777ff5906e0"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/registry.py", "bytes": 6171, "sha256": "b63f7260e8219d36a3c3f4ebd1001c7d729d1f106446b9ca50620f79a2e092c6"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/hud.py", "bytes": 14051, "sha256": "bfc95127fd9594e5e9269071f3d7241875ab33d026316482cb157111a790757c"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/presenter.py", "bytes": 7238, "sha256": "626d70427727c5e0a8ffcf37032323c3516461640694fd9be2968dbabb8a96d4"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/protocol.py", "bytes": 4050, "sha256": "84d85f3e5adbc74feb7d4bedaf96131541cad5ac57012abb86f9cf4b046feb8a"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/summary.py", "bytes": 592, "sha256": "3bba083cf4d57081c50abb1af49307abe586a39ea1ebf2b9891783331085b2fe"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/utils/__init__.py", "bytes": 437, "sha256": "630451edfc5b0032078ed858d971e8520f2ea776a2231aa5b7dc1a9a1958cd78"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/utils/scoring.py", "bytes": 3505, "sha256": "f6c561e79fbfea17be26c526e96d0ed11a6d57edada61ff2cd512f29eceeb69d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/utils/utils.py", "bytes": 4562, "sha256": "df512c616a387afdb69eb5841ad8aa7915f042f9c85ab17594f03bf4a27f2df3"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/viewer/__init__.py", "bytes": 159, "sha256": "e477b6d54612929ba547af4d829ece61a74882c7d2d92c29f76f2e97c370bcf9"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/viewer/templates.py", "bytes": 9661, "sha256": "d7ec71cf42a84cb1c196ad9e7b4bef95611f6353a7d22182b7b5f147d9581b03"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/viewer/viewer.py", "bytes": 27504, "sha256": "16e4eebf920d0de53587c4008d1a0c4b365527940aef2f278c0cfc5fc3c14fc9"}], "required_model": "gemma-4-31b-it-qat-w4a16-ct", "scope": "actual_sdk_mocked_http_and_exact_attached_tokenizer_no_inference", "arms": {"evidence_workflow_v10_cap8192_budget6_v1": {"files": {"agent.yaml": "9c70dec0b2d441d41fc41612fc2bb7b58831f06bc5726cadb6be5b304723fed5", "configs/sampling.yaml": "2a47816e29420420ef12abdefe80787c8759f096dcdc2d6b52307dbad33cb971", "eval_config.yaml": "8959ea845cd35f04dcc3c4e3069629cbe2b9efcfe6ec04999aaf4b7db49d54ce", "prompts/system.md": "8413a77e8f4e271ba134f841665668a59ce0eb82b69f02a2d0249efdc731b98a", "skills/repair-evidence/SKILL.md": "20dfc783a2dcf2f806929e1856e0463c45953b092b8b1ad87295097f8a3fb286", "skills/repair-evidence/scripts/repair_evidence.py": "249a482641ca902981026cdf873f036964fb36a500093be41a16e444c38faa24"}}}, "baseline_arm": "evidence_workflow_v10_cap8192_budget6_v1", "created_utc": "2026-10-06T05:45:09.155552+00:00", "synthetic_response_sequence": ["load_skill", "get_status", "final_text"], "model_path": "/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2", "model_input": "google/gemma-4/Other/gemma-4-31b-it-qat-w4a16-ct/2"}')
SOURCE_BINDING = json.loads('{"parent_notebook_sha256": "63bdeb75c0f711d16b37b6efdfda93f2ad38dbfd19d3ac09685562ef0f8f8fcb", "plan_sha256": "cd3e15c9fec4c93c0b899608ff0a2b25149190b51f4cf04d8b386698281c5fc6", "source_sha256": {"competition/frozen_agent_unpack.py": "d90d71ba52a118cd374a02d15149f01bd7a49dd902ab85a905152b8b2a342062", "competition/chat_template_runtime_probe.py": "5f767927c716c756c296e66c974ec8d88a0d12c4edb7c0fdd12cc14b717c9569", "scripts/prepare_chat_transport_probe.py": "af7ddea33346ef87290c4198f68c79c467f6a0941756c8f544836855f49972c3"}, "candidate_preparation_sha256": "232082ccfc7ce006e8ca825eb64d121e5d5924b7aa52503b15eab61387d5f004"}')
PACKAGES = json.loads('{"evidence_workflow_v10_cap8192_budget6_v1": {"base64": "UEsDBBQAAAAIAAAARl1me0Is2QAAAEUBAAAKAAAAYWdlbnQueWFtbE2QS27DMAxE9zoFewA5CBp04WP0AoIi0TIR/UpSMXz7pjYCdEUOgTczYPUFZ6g42GcnGzpZMWe3NX4suW0mogSmrtTqDN/YPTEw9iakjXcgkYECunIbaQUPSwtDMMJhA2+byZQWMc+QsBRvb/bzerek9ser3W7++mWDGqqiPMIZ9UE15BEROrfSVS6yi2KZSjQJK7JXdKFVxap/c6H0jzkPL8aXnqmmafclG20ty2wALPCoL+pVpcZDJ1Qn6nXIIWXcC6nrXsNq5EH5zZ37hY8/WHxSxBrQ/AJQSwMEFAAAAAgAAABGXZhXMSZUAQAA7wIAABUAAABjb25maWdzL3NhbXBsaW5nLnlhbWytUstuFDEQvM9XlDTnLEmAkF3EORcUkBDnUY+nd6a144fc7bDLiY/gC/kS7AArkfOWZPlgu1yP7vHhkuh6uBj2MusrJZ9WCfPmRH7Frx8/8cCBM5nEgM+UybNx1vri0gq+BiuBJ6hRtqoAytZ23eAxIpG55UqLc6wK8SnHJ/YcDAspRuYAz6Ql87S5uLjO2KcWQqXf4XpzgzN6fPmbGDJ7kqAIMUwtJC9B1MThm9gSi4Gwl2NzyDx1FtOQGtn2Lf5Hj8fiVi6Kf2U893BgTvBRDXGPan+kUVaxEzypPtMddnhzjZfo8VG8GJ6io7GslE/QlJkm7GOuyxWtmuZzy52n41DlpmKDxQMH3eH+ZnvbmD7lmYJ854y5yMTgY1PI72HRaG1DZHy0cxKvb9/d3Xe2SDhUF8OfGdt1gAS3lomHlsu8WP3BcuF6cL47lmlma4a2d91vUEsDBBQAAAAIAAAARl1TeIiOTQAAAF8AAAAQAAAAZXZhbF9jb25maWcueWFtbEstS8wpTSzJzM+z4lJQKMnMTc0vLYkvTk3Oz0sptlIwNjAACucmVsSX5OfnxCcn5uQARU3hgkD18bmZeaUlqUBhM5hoaVEekGsI1AsAUEsDBBQAAAAIAAAARl1zHLL86wYAAFMPAAARAAAAcHJvbXB0cy9zeXN0ZW0ubWSFV8uS2zYQvPMrcMtFku/e08Z2Kqk4Va4t5+yCyKEELwgweOiRr0/PDEhxnVTl5DVFAIOe7p7mC+XoL2TKmUyu8+wdDcblXMkc76Y/23By4WTmFIfaFxeDybGmnowNg+mt9/xrrsfJlW+zLf350P2ZyaQavvVxmvitMSaTaI7ZlZjuhm6zj8nyXjtDgyt5J5tdrHeDPD6YT7Y/m7a+w17ZuGCsCXTlSnrK+b3JxaZirq7gzcG8u8b0mmfbKssz9W684+87VtGAS1G4uBTDRKF02K2cbVmOOJgvlLLLBb/xvgWXJx/DSc4dnSfUGCJ+O5P3qDQ5e8TDQ9f9TMBHqzhR+caLa5YSYqBl/3bYmfrXbE5Oz6jYE5f5Tn3BKbkkxTfvOl5xrM4PZqJigYk9mA8xXAj35T5pe1woEQeEPhHqBarYCNc80tleHCDnXVyYa0GZnyNa5f4mM6Y4LesH3NaNDjc32KndEvvjxjW0WvT/fGgiTxcbGLOBuqX7WHww0vCTwZkn9FkOPsYaGPTMwN96SnPJT8ZeohvM9Rw97UsiMkOdZpT3jPdsQssFxxDNxEwioJ8NXbhQtJVvy3RsXJUF2hVrjim+cp0xevTkj3hpN91wjdfbhbxMOzSo171SxCa9rZmhFRXEBCgP5tlNWNY1pJhNupLLdArL6FJGV3D49iyGBvfFW3IHa2ZvQ2CpWDfsOi56pjShEa2yUwWnD+aFLHCtKQkRtdYjQUBaMjZQtK35ci9nFmOf3Fw6wc0qw2jYmRrcX2hxoVth6XnIgnnPLUItDO5CzScDWvHdmJW170Urh+5Lokzpwp6QEveSa39DrbkeveuZhZRG7I/qP/DxJt9DsTejtEeL3Tgeuo9R9TNTe8yI4lbHOkA25prkclzrajRZYI5VbmBdEqMRSNDil/YibQRxFeqSzQ7ahKSxcnAZvsNabUipNAWnOaJ2NZau4TWgpF5MCqKyrCYBdnotNM2GGZ3MuzLNMKgb9bXI6ZOxJ8sC3phc1wrVhtIN1sLlF8pFxCZbwQTCD3qF1cEu+6qy2a7qZg9fUSFuzNiOgF8BBT/AWHlb/cb6HNHsnMUVhETLi0BMSPgQFy9ouwv50ZmTudoMgt9YCr+NzK/F0a3z8AYRDEPal2p9R2BKEpnGxDjKfoEpaKXYJzMoC3AG0AUIqmfdD/+07dlWv7LE8fcx3lhAcRwxaNTaB2ISoWoHg7C41JyI4Wc3QqGNae0J7t+/2hPlDqXhGZskzh8pybUx+6pyDT+f3YBtN3PIqPV+JssDkruASQU4U/NpmO8ocDfvXH37nTzln92pNktYb8ukYIyNyB/93V6oA+KbQSXI1MTiWhW5NAH8WAaJUt+7ya2EMRAOvP99l0hZrf7H7DU+nnZAd5n6bRblc7yylTbnbq7dJoAcxmLohCm8eGgrmXTiHDmP1bdz96w8b+9rtUNEt7gxE8FFVQjKTXSbLW07PYXUXmdWCwpvid6wVlKPLli/jQ9wB4VKCT1BnQKLmx6xJUsssdtgo74o6YHzjHjzwfxONHeW0UHLBL6He2CE+LuCryHj/Wre5/sccTxCxW4xcr7QMuFFMUv6kX6hvdUvWWijGpEeZ4nHjqo/U66upx0WXhxcp5NePrLZEhIQamBBceT5ZNXHWQXVoxb2996OY/T8HI34BUjlMye/phV+nQMLu7jZ7+Xqu8eTXz89f9xtEg1eEaDkFnDVkiwPJIb2LOOtoBeK2veaJX0M2xaoRDA50TUe4vLmw8Q7JD6HYQPu3GNdTBqDh5W+DaGQaZvOTA2OBQ1NsSc7l6pmuDRCkysj3UKasgYogL+zR8Datf1l2yM3bPHO/RoYOj27pYM22rgXtu/ZBogvQzyIjspNUWzjrKqgWTDG3X715vzqPF+CjU55Pbay9c2WWDcZmjpgiizGkeJzRPtRt11KR14NEiReCEaN4taJw7TaznmcIHB0byMId4cHlUxeiUb98qbK9GrToMKR3qXIZrHQFx7HwKKFGx3J7CMtVOQE75TwInkMDpIQXjmQreElFFV1JxR5mxtkmMJwEVlliA+4H7tpy4FuuO/rzFu38Lhet8Owg9Na9eUfx/b/z1SznakdNKcN0swm+uRW/pQbLzdqXZu96I49T5y3PRB5B9WsSG/1qTdxFCD6xryD+cr6aPRhdDFBM9wVQ2aAZSCmPKH13sP2/+PrT1sigwqjCzMPeJe7QegEGv7OZgGcN1R5GPBmuKzjEv9DjjydHzBottrvZ0mye56h3Ln1AX9n/Cv8vpBMFaYPvgA4NBzvhfKWd8+bWds+2JYhZL18lCGKc58gG0RY2N3j9cd0X9HbeKV+Y6mHs8Mww+TrecvuQ/cPUEsDBBQAAAAIAAAARl3gR9dEvggAAG8TAAAfAAAAc2tpbGxzL3JlcGFpci1ldmlkZW5jZS9TS0lMTC5tZI1YXW/kthV9568g/JIXj9ynooif3KRpFmlTY9cpEBSFwZHuzDCmSIWkxp78+px7L6WRt1u0wGJ3LVHk/Tjn3EPvdjsT3Uhf20yT83lHZz9Q7MkMVPrsp+pT/Nr+QDS1FfZ0mVI9UaFijznNcaDB+mhdX2cX7J4OKZN1cbDuUCnb/kT9S7m1U8Yn+UzY5jBXx/viKa/zsUzUV4tNbZn3o68VW5Y0557wuYtHKp3ZIVLzU8H3c3wuLz6EZw3Qvvp6svqEU/k8Eznj4AM9T66ejH5U7nTV87Kqmy6dfXSlWJeP+At/bPCl2nSwpWYfj6WzTxyhHoooCkeOrw0Hng4H33sUoOC4fXpDNijO8WR9LXY/D0fipOiN+rmm3NkP1fYuxlRRuJ5wLMpCWSpfUpilPLyKX7jBR16S5jrN1XrENpx9SfkiuQ0JreCdqFS3R9An26ecUVL+qjPmU3W5lUmS+9cNFta53Py7s3hXtUQh4VXw6BDO4bw+a8hryi9lcgiQe5717ZIsjVPKDgENng/m0ICJSHN2obzSWubn8586+5cz4X2kV4AiSfIoheTZNkUTOULqLBpuULpnDdgCW7IEawP+Gp2P6Eyr771+zUC41mR00R05FVfbMhTkO2zjViCjngLSKflYb+0MjLUyYQO6ubU3u911Lf/cdd3NrcHjJa31Ia8VgNPw7lmkt7oDRdDW5TmK/yH2YUahlTumQT6kfqFHn8aRe4OO4L9IiX/gLOcIOqVwRl9+ndF3LO/sjwiX8Uv2gp1MH5wfsQmXARX3B4/VB5wFYH0kQAQURRF/Qb/wYlONxuExnbm4KQpNjfSn9UGp7eIFEcaK3IATRIquorIhMVYPy6vOfpskBjCO0AVnC7ncn8xBulAu4z6FtqP2dkrFC4ROrthWTeERvYGRwlZj/rwIzWWRChp85ZJF8HWGnqQINIwOaJD2ckh7Ormzx7n0NgXwtYYLDvh1BmgHs7/I+b6UmTiPMk8ANZ+NN3I0l6MdJo1A4VFLEUcH5BcEDgKSnteUhnlwV8fJnr0T7Wo9RQ/miKxWobiCd8tUMGRPCrbg9hQYPfv5qKiaUCDBnnasPbzUU4o7Vjx+wGffSUCQON0IwCgFcFCY6fdaF92h+pGAOP7hj39gnH6kKYD4yiJ6c+OEJEVNJVRG26L/epIqJUiK4j1KPFBUyD+fZ8cZsnpwPmzovBR66Q+W4QUgTFCGPbIMkEAOxCkS2mmlDgj0lv+lnKUpaFRtMO3s39FMbppnearl1vQpBBIaWnyQMujRssX/yiVW99ZeyGY8mOaAMkEV1wSUYkC0WUcMoPU+j/l4b1uZgZ149jnFkWJlXM3xJabXaF9PhLJM6PCUPaudMd/ItGNZHGaNssFtAck7AD2snFHob5Tr/4ONkO6/9PxphWMmGXbcD/QexCru/KW+7i8ghE62NHkqyqboq4xFJcNmtJh1oNzjiNfshV4yS7M/+oiPhDftyzYt1Q8s1Q4OU/vnNJvRXST9Ff59GvhwyAUQg/a844Vt0nNCt7CIsXXNozOPEM6ILxb1VQvT5o11Z0BXaM67ZDpCiYvK9X5e41xnsVkwLRNrGeV76h3HC32kzKley8gBYiKlOQwtW0xpUAnVMetYO2Si3/C5TFJFN1oyQGEjI/Jyi23e6pxFyLb4OztgDcF35gMrMk8BjNUZbVZ3gSyAjtNm1DQEDv5w+E93p4ZGqvZPhDi4K8OlCoNXX7f2ulu85FK2pboFfiwrC5q0N+uBvKHqLOMLqyVyIXQrEALNrn8h9XlIYY7tgWkPNE7pWsPtw6cnyUlN1zLS0GOdekPqm+1j64rBykPSsOLfYcIc/HFxpuphwWWkqWRkX0kNNYiZKQSWB2FEwgpMWGhnf+rMg4W2cBW5TKN7wYqErk9OVWuVFySOjtkH5gJ/2Thxe41SFcCIuiw1gePerWOLI5eNv2QXnX0HUk4A+MD0jnVX084NoFDEqPwFfVUb0czbolKKWRG0SOzoYAXY3oixN2s3ZGYosqTQpNtuJqduhIFie5AFJVchWty1WQz0V2XjOTs3vDzzG/mc/fEqxAuqglplMJ5phQPREbgQc7UdqnWMs/vWOn56EB2KOgGO62Th+8ErhdC8JNowM7sE/cALknVH8U4ZlY67BrnqCt+EeN8visIhuKNB9AOm3Z5a2OxncG+SCDr7HeZXepUt2ghUu8LJoR/nxbZnCnR2sZrzysp7hCl99/HMSrChBINHwFmwsmBiOYkFHTxsLJEvBhTORTRlva5wOwG6qyr+9PFvt2rfAQgn15Vl4omYphGbkIF2z8XzB3DdlAHWdim07uhYuJvw3KrNUqxrnjNQ2+tnsLN8Obw6zOZTSGwdPtyaCvVVUcwzEw+y4YfLbp5Wi9quoubkBzDPXotn8Y97l7dymPnfr0NbVWIJmJMAnMyqyp5Xyx6FOD1MXAweCoW+KteLrvnEyxUlcmFpZmghJt+JOFiufP38GiORcKpMxWbAjcjxntYr+KB3gqbgMkJWT9E6rmxYLNJ6NZMHRi7ULMh7xzRJ6s0XseI6JPis1bM09tSUAsT7cfk9gNKgVHP12W3USGWVrzICpHF+KQIwz2bljvfDfXVk54Mk1E+Z97PaR4ZbIL5KfiMjRDZ81hMQee8mmZIbQvFQ0BuP/gpkT5ubJtQVYxo/1Atf3GclKZr28O0P+juBkRUdUuV/44S4jEV+PbLRKx6GMmOWu7KK3InCRNnI9Y8nAbDxV1+396HtVaFVdR2sGCWQzSqB41oON8UBPP789P0/fnx8ePr+/tohTUfC+h+XeJm1o5uKvbsexI0RZ+Yj/zqi+TOV9poMpK7QugVPPfZJy02/HcZG8dO7+0FrnHjYVbYNe0qxcr3rT4Jg+BolMjtGbM5avg8CHp2T24LhyqNDoZjmy06skLiIuckP24sjt2BlULHin6/uRubUPXB+dWAuLJNj/dVPu2RnJmAjv92SvzO/A1BLAwQUAAAACAAAAEZdGMlhMakVAACwRQAAMQAAAHNraWxscy9yZXBhaXItZXZpZGVuY2Uvc2NyaXB0cy9yZXBhaXJfZXZpZGVuY2UucHm1PGtz20aS3/UrZpUPAPdIWPblXDk53Cpt7Gxc2c2qbO+9GBYKBIYSIhDAYQCKjEv//foxLzxIK7k7VSUhBjM9PT397kYuLy8/JmW2qQ6LokqTQjSyTvJGyH2eyTKVc/GYt/eirODFVjY4JLKkTUTViPtKtQLWFJskfVDRxcWHrlSivW+q7u4e/ith0l1S5r/KJlBi02V3spWZuHn7o1APeVEIeZBp11ZNJD7IJMtLqZTIlUiyfa6q5nh9kQhEBtak9zJ9EICoXpNsCilklrciaaQoZQpLk+booV1W8E6kVdPItCXIdVNV2+ji8vLyIt/VVYNr7+qkUdI+q/Zi21Q7PKFs850U+oV5ngv8969VaZfcJ+q+yDfm8RdVleZ3pRhYnbQ4xcC6hUczpbFwVH5XJoV96jaALR7Kjhztz1bu6m1e2KWI0sXFRSa3Qt0nId7O7PpCwF8j264pDY4RvH31L695QnQvD1l+J1UbzvTiu7wNH6vmQdUJEvCPQB3VB+TQipquDFcBLAnmIlh8B/9WbeOWz/T69RwYpIbFMq66tu7a5aem02SEgeWrK7M7/JN0RRtbEKHeG+7r+xwuHvkJmLNSOXDMUTzey5JYaQd30+RJAWymaBLzVl7C9SOpqgYZI33MIrx5hIjbAS+0Id6M3gX/8i1xDY5GuYqzvAm9tx4dfkIGcGMK8BZLoh+uBYI0cr8gziLqLNR99bhoq7qQe1kEM39DXh0x4LTK5LM2rIo9CMWSeCnUIFSbAUWjTCKUcAbPTV6Hs1mk54e9fRFZA2iI8A/vbt4Gs9+IE40ZiDSediB7JVKGiI24RnANgNKFRsLMAJlHwiPE6yFIPYchggLIURYVAF3R4YMXlmECYDoaMwISgb7B33SRswjOh5u9EIFbsh7AjeShlWVGyIb7pOjkTGxB1dFP5KkKppT7vKlKhB4Gt//56Ye//3R78+kHpBxQTdUFkBamISsoWc/woAyJRQlogeh/3lYdcDUCx5kI2zserAl5wvXS49WZT6onQ8ZCliGBnYk/LMVLj4RJrqT4N9z8XdNUTRh8l5S4HJVkm2+PAsCIv4AWtRR5A9iArlws7AhoXDhSmrfFUbOuuWzcMqqrmjRIWuC6fwdjAVz4tkqR/co7FYJOjX4CHvrUJKWC0+5ko4UKxRAUfd7G33dl2uZV+VZuQyWLLWrvTM5GvIDvopjXbKrsGNK0C5rGozfqWKYeOCD0aAtv+neI9Jen/a3KOrA3U5PsSXy0Js9AyN/JEpRVypM1+p5U0kiEMMjWwaRStQnYM/didQU6FYn67lA3s55g4t+ZZRFxIS/+rqIZ7e8AYMDABc8G6tFhv3S/Vy+v18OLxJda7zt7HgNivvX6ijAlrRRujiAVM/A4yqrRWr7qGmBOlHNwLiTqKeA3uApgxCbB64l8ZkVQWberwzGHziK+DbcbIQHaJC/TossAsRambjpAYfl9UihpbBYYwxa8C5n5lgRAkICTCqHxiIZ8bMKkPIZ1VFSPElQTyn4YAHQypvhfZX4Agvgzjmk0jgNWRzUuIajuAmHY27JMdpLhplW5xdVRfURQ9ZEe8jKnPaqD+Vkf4Ti/wHGittoVwXzEGb2/QMFJ6ijdEn78wBs08r878Nl2oGFU1B7a4ByKYKXwFOhjMgXghJa6SZveg6TIR8+vMDSGV9rkei5LkOXbLZvcTV6C2XfWjJbhjursMpyxqMqCVi5+HazvyrYBV5dM7xBGoRbIi4rhVMCijf4tD8RFC5SoLGkyDZpBPt7ncGzSs+fwIv93gA3oC2Sjg2eoiTkOdO9Eojkfee4w90V2wjxUXZGRgQEFUAMvkKgladtBZNCzExqH9D4p74geq4NxPYKu3S6+0ZxKyBASxkFhA7kJfr4KyDYe1pq2D2X1WH4Bkj3HF6B9JT5h+LHdguFKXKDRdxbR+DkPUaXgL4Gr36FWgHODpcXYJtLwPkh2tnEcblTWqGI82/gG3M0S9M4edRK48gIvT+w6CJFAZVRiIzmIAdQJokEpJq6hY/vHZGKQgxhtu6LY4XWGTfBzlGQPMS6OV1eLf00W2/Xnb55+Ztk7zMa0nAR60Hc8wIJXg4YAPRvz1Sq0Y2U8HJPIL4Q1WCP+h9biZkb1aN5w/AaiDj4TLHK+xQuaPfTAeaJzXNEbb2RB1I3baqQOzB9jFSV1jT7c5wD1THBNW4Do0Ft4DFAHgdOU7IHujqGf+rYQ1CZo366PG2osAK31FRJ9gEHV5Hd0+2NhxiDASPB1IP6JoPX3bDHmHWrajQSionYYWEqzlZYE4n475qmE5VJcCQlWq++xm78EjEMzhm6vIMliMr7hjHbQ43AhyDFgu05D7vPMyVvp7atnZ/Benxs8WkLxC/bI/QXWKgMQZ6GJ2oM7ZjkW4cdj2SYH0oFz8feP+odTjDNwH5i5xtfzPJ5rj7UM6WEWxTG+jGMPGSTi0JUaCp3ZwrGNdiZoT7CUHN7DZpgFoCHDGmjdaQrdJMzAiKE/w24e6BuItaKNKY6BNXocYFklbN9p1eJBadHdz8EHiI0Sjq2uMav6ysdb3D84XuNA/QRj8iDNx4rKwdQGDZR2zFeGCNAPbWhjUs+AJXjxKDjw3hlnT6B8PA13eUj4GtdYx9x3FUFJ+9QGPVTGsCLeJErGMG0j6Sit8C+I3HIc1JR+0j5SUYF8gmPRSl/X0ECM8ZlWT07hesrX6ld0hFPOzDBjufUw3z0MMglwKn+ir9aBAN6yZ2jvsSvyTmfy2ISC9QQqKBhi57/b7PIWk4lDh+QeFNaE8p3Ib5gz4IrJXMcYpw/s2iow9egM4Y0VeWm2No+wPcE8kZDhfSkAbzFQQmkN+8kzMA7BzyXaCANzFlGEA2Bmq+tXX7OxzXLMbWIyzL8lsKkGen9WtHvATAinQZROxAG7g9atHka3T0mJpbfHCzBh+CbCHGfQ5wBKmREo5afM6CUAwRURsqoKvfk92+KbWZqzCszZQRp3cNvBGm2BJbJhsZWfzcEZfVIOUlen2WwjiwqiQdFWcLuoDyRliwZ33FfU5nyfAwXKY5fEe3D6QcOAAL+a+2mm61F2dHS6a7vVhKUDfceKG12tgJSEfWpM6jwGOTOpc3iJF/rk2wl7lXPv1vRvrU42mHICHdXKQzsXRQ6ILb++urrSZNS5Jnw7E98uecIoS4OvLwbPq2uaK168EK/WzN4rTgiD09OVKaCQvRHo6IJKu8OVCZwpW7MUEISFg3C9dgH4RpL50IzERxpe/1fiRujEtVBc7YAtHhvU7iB/EnbFW4A3igsHBAxjIFQ4VK+g4EC5YIAUocgh0s+qtMNgF5TRPm8oUiIDJzqFgYQOntLHjHQ4bJ+3EH1k4tPfbt++/xCZePa+lzLoZRJIWhIVY+77EJoUY/DC6TC255gOEH8Sr2gjegRdsUbNvAoo2N/Vwdpdl7/piZTpL1Ve4rTwjwwOKG/k4Nm7euI52tvZopM7aSYaaQzNAcA8bDR9dT/k9Lnwahk6xhjEVatk8evN4r8gpIoX688v56+/fgp4WVQkoBvO2qp/wHMCjEQ1pKIGqw8cATaV+VMQABMw26T4ZHqDDYnx+Evg+zBsyKFo0KEAMwYaBgRUqz6tCdbjNB4ftFkFvDndhzsN3RW8BFSV5JcB74pJdPTmXYoBF9E8f9aoaKLPtZp2aNZfUsMfwEg2XOPBZbrepwmxSx5Qkpx7Ybwtd2zYBhU5mn7V7cKDnyf3kiJ0GC3q+naNw4ieXW8ExaFvl+wm/Sz79Ilu+CDMAaWUGaYNQBkUnHS3SNzSboJCNcyP8SOlNXub+6hPF0xILsmlxswEWiUgGwZB4LArmRTopvJ6FEvz87oH+elMSFInR7ThADkcUs54JzZhYxnHzfGRPhXLUTA5UurDW/EVfJ/zbQWPMQV6so6iR9RSr9GajYO48fVh+oiJZq4kUUo2RFrySDE/ADZKvIwiAso2IxgmEXY1hskaAyD7twx00Ug4ZtZRQPKnQMfBg9U6xd13wrbBZzwTK4BLVgCX69n11dfZU0SPmJaYgBORvbN2kknyDObRrMP0p2QvgaObQMeGgVO8ezpEN4sGMSoj4WD241N++zTheME9a90ATDWWgvF1/lT54kgtDZTJ81XzkAS8wyqwY8H6OQJJujW0i1ZMSHJNx4L43KQGsnJvwz8shbeFAbf+or97Q+keVq/Ex2BASe2ScgL3ZGd104T+OSHOFjtPSE/rKNXV4MDI/1dF8n+hSaavsC8OFIYAK5szzQYX02f79XM0z/iGZF0kR457WSM5VeQrHf1yOYH4coS4jd9o1AtppnTO+EAk/OuhE6B1gUsOGurweD+Z+BsINSbSx0lCmITLGyylk1cC+hcbhLhFwtfOX4lFKh6krJVf4vD6SwCmOnIVH/NAwNAxc1Jsque5VB44up5k7yMVAYvq8EIeaowukCA28HDIY3wTWViPDWb6MCcbBrrFR7cQzQGjN6Je6seIXHfEEvh6v3q5nr0RkxqFVFhojVHvIuYYmdaz+SXOuZzNpwF8vjRZy8tr+LkDywc/55eGJpfXDOZp1qPx+7sSlXTdyAVlB6iyku8kWK9dvUCNnAmQx1RwEYRz0Qncd51n1N4VecBuxBbu8d72eOmGNQC+zQ+kdRd/Fsm+gqUcE9+9QErjnnivwBMgVMpBNKoUAhWkoUtEY8Xtz1R3+w8ujNLKmDdaYkCKh+1lRjCEhTlZzBcGnDZpDINFChD1BXPLFDO+Ltz0rZxDcErV+7GfHkQP5+WrnoczFp1bMoGe4WqrCuxgc2ccTirJkiLBq4p2FUhSVeapTl3hYIbdXDCBquEcq+rweuk3id1WtUNuzqKwJEVB4jvHqHjp4rYvGUNOqC39Dd7fvkMyZrJpRuNfBgfnjEv5GCtYA8RdVorr5qguKdjWFOnVZRgLsysGcnpTPGcHdMJkrOlvI4OjH5xg+NUZu9wN6vog1Sa8Q31iMO8ONbZE9pyhMeKDspSKsCmuvgsNMBCwuW46jD6+/8uP7//613PlCLMMoXiNZM+ixdS55z02Qks+p4SVZ8hYQcUgWbbGHBr7h4mUga05f9u04pQlWp6zRLryX2NfVnY2bIegf0Nq2/nngwCdyWXzbkgzk2cFt57XsyPPM0FXjMdhEevongbyQM96nr4u8EyuMPuPVsALk31JJVbFMSBgX/naSyOgcqQMwrWXJrA1PU6E2t9+0YM1Qj8aoCQxtugoq85c0pvynIZ5YMFkOcYa1NjcIbELwmd946ZqFnMrLJsBEWwVZpoR/SqXYWIs8ZnfWE/quAUJMIA4MVP6ta9LxcIo2l41CDACzyLl1dzMxDgld+B02Gi9N88DwDFLPCgI2uRMbxyTx2Rxh9M1s4+mu20s50/TLTyxIUraCeDPDoRQki34YSWyv8PorV/mpOzUY6JiSlj5RJrOYPnMw7LW415XadVyNXyLIjWGYYJoXzEYCO6dkVOfU0CFYeGxa1OYY/rTo7J6DE2LegTvsHWiwq7LpA11JD3QTKasrCV9UFnWoyRx9GtuUMeCha4W9AtdXFRdBhS2YF1u4Arp4/XXw8Dp9abYaSsdoc7q6ivzEr62PyzX3Sn0iAlA9i3zcnj+nh2lSS4tivEeDQ3M6ag3BFzDfV51tKWXrh1spTOyE0nXSe7XOVo/gcvomRE9wWeFtfjWzOoNj0I1g/EXDmZT0Wb+avHSwXputuR8UMmN1H5gSQX+dpEsdLKMRdCZeH3vnvomVkIP4Tmx7+/wH9yN/G+8CE3/0NBqYAf9Ox6+GqHn6cEhHSbnmj09M7o2XWDhlU7600yNQW8iNhGdBftFy2X4/T4BPmrGHO8DO2XI1z5+JyedBf0cwz8NQW/7+wEYFAaWZ917edaynkXs/ErbQGI2cq7LNFT6foohe1N7FzD2HeiSTxj/2bCakLu2qb5i0yVG96EXatUVd1atnXZ1+/QbgaZ8Cda8J3rNHHGaldc9ppUmSDl4pp41sUUGOIBf/KZp5kSYcTa1vm2SFwum+wv8umLBtsjIAuUwbFraxTtnk30aCXeis5iYtmVdyvBIe/Pxk86avbE9vyptqL8aCG2K2yTV1BhNaRkPC3cLo2ao9Wmcbs1kQY3tuBX2qed32nE2mbyJfUZe3eld/mH7tbkinyZw9hJp328iIkZ94/c87yY2HneQrQlv836yiewMdtx1ODef/xjgRHbsHeYUMDKPpYP1yxzvxCYdFqM5OureMb0Xt4fgL/LgeMw5kfxxJauImISPmu2QsfudmaNWOy9Z5sBRgwa+PZWezSphsj8CK9lH/U0kTM0kEgXOccSeJLgibKLd5gf8dBApoj+6Ql4U+6TJcetI3KBpWRhjAycs5B4/W2Hr2DNJ9KEo8k6nYL9IfMLyT1ZJpd2hau/1i1OjG1nyN30R0JqiXGhJahP1oKi8C6eq8cMTKX6BHTA7jPhgniAKjAOLaVOsQeyXZG3t9yKK0gbma9ToprmjHW/pTZhJ7j7BDFUcAzpxPPNWRkkG5kgvCQPvyy3wo7DvdHnrd5RMrSA/dQHOmb9ibj7JXJ7pEMHcZylBaAv1KB0r7r8xucRuQzkTuy3ms+hJ4bnaZVDVkkUe9tafjmR+M1q3oXW8KKQWtM6oH+yK4oxjbw4OezNGx70/1vh9hsrV9J6Ti8zZfsOSRm47NCSOksE5tLDbY5Gkp2nBVmJ8Xt895grwEDQb1tNARws4RoEI677KU6mWoYlUTM4A+0SeDa1fUDyxIwnUgpTmM+bpOGHI39Pb9xTB4FTaH0V951RMQF3O1GmLB7XXZ8fO7KazvQa5nOJTDeD1Fa/EJI6TC/oPwqAC5V5/GGTdOQNc6A4e+8ZYqImpuiyOmjMcv/aaycgr976hta7QxCrzTbTnyU1iOfEpNy0414CIadOJXmYLVvfm8DoANCCS6WMbt4Qaw6k/0v5sv5oleFb7EB1IcfQTAuUxxHTunkorr6+u+Gtg10zkdInG0P0fCOhR6wD9hCIes4gPv50cF2p+lLIWMkldxQsMoiwygVmThrEBiwweTovfmvlZeSIC94yu3acJnt7TeUQf+6DaADPuwR+wqu56cKR+HkcfLR6DtYcOvAObl97QAODvyWnNdVcyfW44mTClPliva3Hi3nXecpgjeYk9rn4FB5//edg/dKrCZtaY/nX07rlfCEAIkxj2iyOaRX9LSyOXp+w3bEtPsCLcEOwt+DSHMACyPwaD6Trrj9VnP/uu4efojrXLV6YbfW4/sl2a7owBOJ2x83q8n9/zqP8nBpy7Da0jYY+97JVcyfikmPogJl/2OX61eH29nir/saObVuAGLqeKNHNGcGnzijbjuHxO7tFYBVD5Pj35aB5B+1/R01twEIHpTI2dmNJU2TVfku84u/gfUEsBAhQAFAAAAAgAAABGXWZ7QizZAAAARQEAAAoAAAAAAAAAAAAAAKSBAAAAAGFnZW50LnlhbWxQSwECFAAUAAAACAAAAEZdmFcxJlQBAADvAgAAFQAAAAAAAAAAAAAApIEBAQAAY29uZmlncy9zYW1wbGluZy55YW1sUEsBAhQAFAAAAAgAAABGXVN4iI5NAAAAXwAAABAAAAAAAAAAAAAAAKSBiAIAAGV2YWxfY29uZmlnLnlhbWxQSwECFAAUAAAACAAAAEZdcxyy/OsGAABTDwAAEQAAAAAAAAAAAAAApIEDAwAAcHJvbXB0cy9zeXN0ZW0ubWRQSwECFAAUAAAACAAAAEZd4EfXRL4IAABvEwAAHwAAAAAAAAAAAAAApIEdCgAAc2tpbGxzL3JlcGFpci1ldmlkZW5jZS9TS0lMTC5tZFBLAQIUABQAAAAIAAAARl0YyWExqRUAALBFAAAxAAAAAAAAAAAAAACkgRgTAABza2lsbHMvcmVwYWlyLWV2aWRlbmNlL3NjcmlwdHMvcmVwYWlyX2V2aWRlbmNlLnB5UEsFBgAAAAAGAAYApAEAABApAAAAAA==", "sha256": "0f9cd9c67c93525c03cbd5718d011031c12bd6a9a1fb7bbf65a79835a9b5071a"}}')
PLAN_BYTES_BASE64 = '...'
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    RUN = Path('/kaggle/working') / ('chat_transport41_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
    RUN.mkdir()
    frozen = base64.b64decode(PLAN_BYTES_BASE64, validate=True)
    if hashlib.sha256(frozen).hexdigest() != SOURCE_BINDING['plan_sha256'] or json.loads(frozen) != PLAN:
        raise ValueError('Frozen plan changed')
    (RUN / 'plan.json').write_bytes(frozen)
    (RUN / 'source_binding.json').write_text(json.dumps(SOURCE_BINDING, indent=2) + '\n', encoding='utf-8')
    export_run(RUN)


In [ ]:
"""Fail-fast, recorded CPU setup for the unchanged organizer source audit.

Definitions do not install packages. Runtime checks never filter incompatible
wheels into a superficially successful setup or establish scoring equivalence.
"""
from datetime import datetime, timezone
import glob
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import sysconfig
import traceback


REQUIRED_CPU_RUNTIME = {
    "implementation": "CPython", "python_major_minor": [3, 12],
    "system": "Linux", "machine": "x86_64",
}


def environment_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while chunk := stream.read(1048576):
            digest.update(chunk)
    return digest.hexdigest()


def environment_package_versions(names):
    values = {}
    for name in sorted(names):
        try:
            values[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            values[name] = None
    return values


def cpu_runtime_snapshot(package_names):
    return {
        "implementation": platform.python_implementation(),
        "python_major_minor": list(sys.version_info[:2]),
        "python_version": sys.version, "executable": sys.executable,
        "executable_realpath": str(Path(sys.executable).resolve()),
        "system": platform.system(), "machine": platform.machine(),
        "platform": platform.platform(), "libc": list(platform.libc_ver()),
        "soabi": sysconfig.get_config_var("SOABI"),
        "multiarch": sysconfig.get_config_var("MULTIARCH"),
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "package_versions": environment_package_versions(package_names),
    }


def cpu_runtime_errors(runtime):
    return [key + " must be " + repr(expected) + "; observed " + repr(runtime.get(key))
            for key, expected in REQUIRED_CPU_RUNTIME.items()
            if runtime.get(key) != expected]


def normalized_organizer_wheel_name(name):
    # Retain the organizer workaround already used in frozen notebook 10.
    return name.replace("cu128", "+cu128") if "cu128" in name and "+" not in name else name


def classify_organizer_wheels(paths, supported_tags=None):
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename
    supported = set(sys_tags() if supported_tags is None else supported_tags)
    rows = []
    for path in sorted(map(Path, paths), key=lambda item: item.name):
        if not path.is_file():
            raise ValueError("Missing organizer wheel: " + str(path))
        row = {"source_path": str(path), "original_name": path.name,
               "install_name": normalized_organizer_wheel_name(path.name),
               "bytes": path.stat().st_size, "sha256": environment_file_sha256(path)}
        if "cutlass" in path.name.lower():
            row.update(selected=False, compatible=None,
                       exclusion_reason="existing_organizer_cutlass_workaround")
        else:
            try:
                distribution, version, build, tags = parse_wheel_filename(row["install_name"])
                row.update(selected=True, compatible=bool(tags & supported),
                           distribution=distribution, version=str(version),
                           wheel_tags=sorted(map(str, tags)),
                           matching_runtime_tags=sorted(map(str, tags & supported)))
            except ValueError as error:
                row.update(selected=True, compatible=False, wheel_parse_error=str(error))
        rows.append(row)
    if len({row["install_name"] for row in rows}) != len(rows):
        raise ValueError("Ambiguous organizer wheel aliases")
    return rows


def remove_recorded_cutlass_hooks():
    removed = []
    for pattern in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth",
                    "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
        for name in sorted(glob.glob(pattern)):
            path = Path(name)
            row = {"path": str(path), "sha256": environment_file_sha256(path)}
            path.unlink()
            removed.append(row)
    return removed


def source_audit_import_check():
    import torch
    from swegemma.config import EvalConfig
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    count = torch.cuda.device_count()
    if count != 0:
        raise ValueError("Select Accelerator None: source audit requires zero CUDA devices")
    return {"torch_version": torch.__version__, "cuda_device_count": count,
            "required_organizer_imports_succeeded": True}


def prepare_source_audit_environment(wheelhouse_dir, run_root, expected_packages,
                                     checkpoint_exporter, source_binding):
    """Install the whole selected organizer wheelhouse only after compatibility.

    Always retain runtime identity and setup failures before allowing the exact
    source inventory. UI labels are recorded separately from verified runtime.
    """
    run_root = Path(run_root)
    wheelhouse_dir = Path(wheelhouse_dir)
    run_root.mkdir(parents=True, exist_ok=False)
    package_names = set(expected_packages) | {"torch", "vllm", "pip", "packaging"}
    report = {
        "scope": "CPU_source_audit_environment_setup_not_model_or_grading",
        "created_utc": datetime.now(timezone.utc).isoformat(), "status": "started",
        "required_runtime": REQUIRED_CPU_RUNTIME, "source_binding": source_binding,
        "before": cpu_runtime_snapshot(package_names), "after": None,
        "wheelhouse_path": str(wheelhouse_dir), "wheel_manifest": [],
        "selected_wheels_omitted_for_incompatibility": [],
        "cutlass_hooks_removed": [], "pip_command": None, "pip_return_code": None,
        "model_inference_executed": False, "source_inventory_executed": False,
        "tests_executed": False, "private_scoring_equivalence_established": False,
        "runtime_image_id_verified": False,
        "requested_UI_profile": "original 2026-07-01 image, Accelerator None, Internet off",
    }

    def save():
        (run_root / "environment_receipt.json").write_text(
            json.dumps(report, indent=2) + "\n", encoding="utf-8")
        print("MOBILE_SOURCE_AUDIT_ENVIRONMENT=" + json.dumps({
            "status": report["status"], "receipt": str(run_root / "environment_receipt.json"),
            "python_version": report["before"]["python_version"],
            "runtime_errors": report.get("runtime_errors", []),
            "incompatible_wheels": report.get("incompatible_wheels", []),
            "pip_return_code": report["pip_return_code"],
        }), flush=True)
        checkpoint_exporter(run_root)

    save()
    try:
        if not wheelhouse_dir.is_dir():
            raise RuntimeError("Attach the official organizer wheelhouse before setup")
        report["wheel_manifest"] = classify_organizer_wheels(wheelhouse_dir.glob("*.whl"))
        selected = [row for row in report["wheel_manifest"] if row["selected"]]
        report["runtime_errors"] = cpu_runtime_errors(report["before"])
        report["incompatible_wheels"] = [row["original_name"] for row in selected if not row["compatible"]]
        if not selected or report["runtime_errors"] or report["incompatible_wheels"]:
            report["status"] = "preflight_rejected_before_install"
            raise RuntimeError("CPU source-audit preflight rejected this environment; "
                               "use a verified Linux x86_64 CPython 3.12 worker. "
                               "Incompatible organizer wheels were retained and no install ran.")
        report["status"] = "preflight_passed"
        save()
        links = run_root / "organizer_wheel_links"
        links.mkdir()
        for row in selected:
            (links / row["install_name"]).symlink_to(Path(row["source_path"]).resolve())
        report["cutlass_hooks_removed"] = remove_recorded_cutlass_hooks()
        command = [sys.executable, "-m", "pip", "install", "-q", "--no-index",
                   "--no-deps", "--force-reinstall"] + [str(links / row["install_name"]) for row in selected]
        report.update(status="installing", pip_command=command)
        save()
        result = subprocess.run(command, capture_output=True, text=True)
        (run_root / "organizer_install_stdout.txt").write_text(result.stdout or "", encoding="utf-8")
        (run_root / "organizer_install_stderr.txt").write_text(result.stderr or "", encoding="utf-8")
        report["pip_return_code"] = result.returncode
        importlib.invalidate_caches()
        report["after"] = cpu_runtime_snapshot(package_names)
        if result.returncode != 0:
            report["status"] = "organizer_install_failed"
            raise RuntimeError("Organizer wheel installation failed; inspect retained install logs")
        observed = {name: report["after"]["package_versions"][name] for name in expected_packages}
        if observed != expected_packages:
            report["status"] = "organizer_core_identity_rejected"
            raise ValueError("Installed organizer core packages differ from the frozen source-audit plan")
        report["status"] = "checking_CPU_organizer_imports"
        save()
        report.update(source_audit_import_check())
        report["status"] = "ready_for_unchanged_source_audit"
        save()
        return report
    except Exception as error:
        if report["status"] in {"started", "preflight_passed", "installing", "checking_CPU_organizer_imports"}:
            report["status"] = "setup_exception_before_source_audit"
        report.update(error_type=type(error).__name__, error=str(error))
        (run_root / "setup_exception.txt").write_text(traceback.format_exc(), encoding="utf-8")
        save()
        raise


In [ ]:
def resolve_organizer_audit_api(evaluator_module):
    verifier = getattr(evaluator_module, "verify_task", None)
    if not callable(verifier):
        raise ValueError("Installed Evaluator module has no callable official verify_task")
    implementation = inspect.getmodule(verifier)
    namespace = getattr(verifier, "__globals__", {})
    apis = {"verify_task": verifier}
    for name in ("resolve_task_snapshot_paths", "extract_snapshot", "sandbox_start", "sandbox_exec", "sandbox_stop"):
        # verify_task's own globals are the callables used by its retained source.
        # Snapshot resolution can be a public reexport from Evaluator's module.
        function = namespace.get(name)
        if function is None and implementation is not None:
            function = getattr(implementation, name, None)
        if function is None and name == "resolve_task_snapshot_paths":
            function = getattr(evaluator_module, name, None)
        if not callable(function):
            raise ValueError("Official verifier implementation lacks callable " + name)
        apis[name] = function
    placeholder = object()
    calls = {
        "resolve_task_snapshot_paths": ((placeholder, "instance_id", "repo"), {}),
        "extract_snapshot": ((placeholder, placeholder, placeholder),
                             {"patch_path": placeholder, "base_snapshot_path": placeholder}),
        "sandbox_start": ((placeholder,), {}),
        "sandbox_exec": ((placeholder, placeholder, "command"), {}),
        "sandbox_stop": ((placeholder, placeholder), {}),
    }
    for name, (args, kwargs) in calls.items():
        inspect.signature(apis[name]).bind(*args, **kwargs)
    for name in ("sandbox_start", "sandbox_exec", "sandbox_stop"):
        if not inspect.iscoroutinefunction(apis[name]):
            raise ValueError("Official sandbox callable is not asynchronous: " + name)
    return apis


In [ ]:
"""Recorded CPU/GPU preflight for separately frozen future public comparisons."""
from datetime import datetime, timezone
import hashlib
import importlib
import inspect
import json
from pathlib import Path
import subprocess
import sys
import tempfile
import traceback



def declared_backend_check(mode, torch):
    count = torch.cuda.device_count()
    names = [torch.cuda.get_device_name(index) for index in range(count)]
    if mode == 'cpu_controls':
        if count != 0:
            raise ValueError('Select Accelerator None for fixture controls')
    elif mode == 'gpu_comparison':
        if count != 4 or any('L4' not in name for name in names):
            raise ValueError('The declared comparison requires exactly four NVIDIA L4 GPUs')
    else:
        raise ValueError('Unknown declared experiment mode')
    return {'cuda_device_count': count, 'gpu_names': names, 'torch_version': torch.__version__}


def prepare_declared_environment(wheelhouse_dir, run_root, plan, checkpoint_exporter, mode):
    run_root, wheelhouse_dir = Path(run_root), Path(wheelhouse_dir)
    receipt = run_root / 'environment_receipt.json'
    if receipt.exists():
        raise FileExistsError('Preserve the existing environment attempt')
    expected = plan['expected_core_packages']
    names = set(expected) | {'torch', 'vllm', 'pip', 'packaging'}
    report = {'created_utc': datetime.now(timezone.utc).isoformat(),
              'scope': 'declared_public_comparison_environment_not_model_or_control_results',
              'mode': mode, 'status': 'started', 'before': cpu_runtime_snapshot(names),
              'after': None, 'wheel_manifest': [], 'incompatible_wheels': [],
              'selected_wheels_omitted_for_incompatibility': [], 'pip_return_code': None,
              'plan_sha256': environment_file_sha256(run_root / 'plan.json'),
              'model_inference_executed': False, 'fixture_controls_executed': False,
              'runtime_image_id_verified': False}
    def save():
        receipt.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
        print('DECLARED_ENVIRONMENT=' + json.dumps({key: report.get(key) for key in
            ('mode', 'status', 'runtime_errors', 'incompatible_wheels', 'pip_return_code')}), flush=True)
        checkpoint_exporter(run_root)
    save()
    try:
        if not wheelhouse_dir.is_dir():
            raise ValueError('Attach the complete official organizer wheelhouse')
        report['runtime_errors'] = cpu_runtime_errors(report['before'])
        report['wheel_manifest'] = classify_organizer_wheels(wheelhouse_dir.glob('*.whl'))
        selected = [row for row in report['wheel_manifest'] if row['selected']]
        report['incompatible_wheels'] = [row['original_name'] for row in selected if not row['compatible']]
        if not selected or report['runtime_errors'] or report['incompatible_wheels']:
            report['status'] = 'preflight_rejected_before_install'
            raise RuntimeError('Use a compatible Linux x86_64 CPython 3.12 organizer worker; no incompatible wheels were omitted or installed')
        report['cutlass_hooks_removed'] = remove_recorded_cutlass_hooks()
        # Link staging stays outside the exported run; its wheel identities remain
        # in the receipt. It cannot collide with strict artifact link rejection.
        with tempfile.TemporaryDirectory(prefix='neural_swe_declared_wheels_') as temporary:
            links = Path(temporary)
            for row in selected:
                (links / row['install_name']).symlink_to(Path(row['source_path']).resolve())
            command = [sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                       '--no-deps', '--force-reinstall'] + [str(links / row['install_name']) for row in selected]
            report.update(status='installing', pip_command=command)
            save()
            result = subprocess.run(command, capture_output=True, text=True, timeout=1200)
            (run_root / 'organizer_install_stdout.txt').write_text(result.stdout or '', encoding='utf-8')
            (run_root / 'organizer_install_stderr.txt').write_text(result.stderr or '', encoding='utf-8')
            report['pip_return_code'] = result.returncode
        importlib.invalidate_caches()
        report['after'] = cpu_runtime_snapshot(names)
        if result.returncode != 0:
            raise RuntimeError('Organizer installation failed; retained install logs are available')
        if {name: report['after']['package_versions'][name] for name in expected} != expected:
            raise ValueError('Installed organizer core identity differs from the frozen plan')
        import torch
        from swegemma.config import EvalConfig
        from swegemma.evaluate import Evaluator
        from swegemma.models import load_tasks
        report.update(declared_backend_check(mode, torch))
        apis = resolve_organizer_audit_api(inspect.getmodule(Evaluator))
        verifier_sha = hashlib.sha256(inspect.getsource(apis['verify_task']).encode()).hexdigest()
        report['verify_task_sha256'] = verifier_sha
        if verifier_sha != plan['expected_verify_task_sha256']:
            raise ValueError('Actual organizer verifier differs from the frozen contract')
        report['status'] = 'ready_for_' + mode
        save()
        return report
    except Exception as error:
        if report['status'] != 'preflight_rejected_before_install':
            report['status'] = 'setup_exception_before_experiment'
        report.update(error_type=type(error).__name__, error=str(error))
        (run_root / 'setup_exception.txt').write_text(traceback.format_exc(), encoding='utf-8')
        save()
        raise


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    prepare_declared_environment(WHEELHOUSE_DIR, RUN, PLAN, export_run, MODE)
else:
    print('No organizer installation executed locally')


In [ ]:
"""Unpack exactly the archive and members declared by the frozen study."""
import base64
import hashlib
import io
from pathlib import Path
import zipfile


def unpack_frozen_agents(run_root, plan, packages):
    for arm, package in packages.items():
        payload = base64.b64decode(package['base64'], validate=True)
        if hashlib.sha256(payload).hexdigest() != package['sha256']:
            raise ValueError('Candidate package differs from frozen bytes')
        expected = plan['arms'][arm]['files']
        destination = Path(run_root) / 'agents' / arm
        destination.mkdir(parents=True, exist_ok=False)
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            names = archive.namelist()
            if len(names) != len(expected) or set(names) != set(expected):
                raise ValueError('Unexpected candidate package members')
            for name in names:
                relative = Path(name)
                target = destination / relative
                if relative.is_absolute() or '..' in relative.parts or not target.resolve().is_relative_to(destination.resolve()):
                    raise ValueError('Candidate member leaves the submission directory')
                data = archive.read(name)
                if hashlib.sha256(data).hexdigest() != expected[name]:
                    raise ValueError('Candidate member differs from frozen bytes')
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(data)
        (Path(run_root) / (arm + '.zip')).write_bytes(payload)


In [ ]:
"""Actual ADK/LiteLLM HTTP serialization and attached Gemma tokenizer, no inference."""
import ast
import copy
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path


def vllm_argument_postprocessor():
    """Extract the installed pure function, without importing GPU vLLM modules."""
    dist = importlib.metadata.distribution('vllm')
    source = Path(dist.locate_file('vllm/entrypoints/chat_utils.py'))
    raw = source.read_bytes()
    tree = ast.parse(raw)
    function = next(x for x in tree.body if isinstance(x, ast.FunctionDef)
                    and x.name == '_postprocess_messages')
    module = ast.Module(body=[ast.ImportFrom(module='__future__',
        names=[ast.alias(name='annotations')], level=0), function], type_ignores=[])
    namespace = {'json': json}
    exec(compile(ast.fix_missing_locations(module), str(source), 'exec'), namespace)
    return namespace['_postprocess_messages'], {'version': dist.version,
        'source_sha256': hashlib.sha256(raw).hexdigest(),
        'function_ast_sha256': hashlib.sha256(ast.dump(function).encode()).hexdigest(),
        'scope': 'Installed argument postprocessor only; no vLLM server or full request parser'}


async def probe_chat_transport(run_root, plan, packages, model_path, checkpoint_exporter):
    root = Path(run_root)
    target = root / 'chat_transport_probe.json'
    report = {'created_utc': datetime.now(timezone.utc).isoformat(), 'status': 'started',
        'scope': 'Actual compiler, ADK Runner, LiteLLM and mocked HTTP; attached tokenizer only',
        'model_inference_executed': False, 'grader_executed': False,
        'stage1_loader_executed': False, 'weights_loaded': False,
        'synthetic_model_responses': True, 'actual_http_socket_used': False}
    def save():
        target.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
        checkpoint_exporter(root)
    save()
    manager = client = runner = None
    try:
        os.environ.setdefault('LITELLM_LOCAL_MODEL_COST_MAP', 'True')
        os.environ.setdefault('LITELLM_TELEMETRY', 'False')
        from adk_submission import compile_submission, VllmConfig, VllmServer
        from google.adk.runners import Runner
        from google.adk.sessions import InMemorySessionService
        from google.genai import types
        from openai import AsyncOpenAI
        import httpx
        from swegemma.config import build_submission_limits
        from swegemma.context import SwegemmaContext
        from swegemma.sandbox import SubprocessManager, AdkSandboxCodeExecutor
        from transformers import AutoTokenizer
        report['actual_package_versions'] = {name: importlib.metadata.version(name) for name in
            ('swegemma', 'adk-submission', 'adk-eval-core', 'google-adk', 'google-genai',
             'litellm', 'openai', 'httpx', 'transformers', 'vllm')}
        for row in plan['expected_organizer_python_sources']:
            dist = importlib.metadata.distribution(row['distribution'])
            source = Path(dist.locate_file(row['package_relative_path']))
            if dist.version != row['version'] or hashlib.sha256(source.read_bytes()).hexdigest() != row['sha256']:
                raise ValueError('Actual organizer source changed')
        report['all100_organizer_sources_match_completed_inventory'] = True
        unpack_frozen_agents(root, plan, packages)
        manager = SubprocessManager(base_dir=root / 'synthetic_sandboxes')
        manager.initialize({'fixture.txt': 'Transport-only synthetic fixture.\n'})
        context = SwegemmaContext(docker_manager=manager, container_id=manager._default_sandbox_id,
            max_tool_calls=50, max_time_minutes=6, max_exec_seconds=300)
        context.start_agent_session()
        executor = AdkSandboxCodeExecutor(sandbox=context.sandbox, timeout_seconds=300,
            start_time=context.agent_start_time, max_time_minutes=6, budget_check_fn=context.check_budget)
        alias = plan['required_model']
        server = VllmServer(VllmConfig(model=alias, host='127.0.0.1', port=8000,
            max_model_len=32768, tensor_parallel_size=4, tool_call_parser='gemma4', reasoning_parser='gemma4'))
        models = server.create_model_registry(aliases=alias, model_prefix='openai/', api_key='EMPTY')
        limits, constraints = build_submission_limits()
        agent = compile_submission(root / 'agents' / plan['baseline_arm'],
            tool_registry=context.create_tools(), model_registry=models, code_executor=executor,
            script_timeout=300, limits=limits, generation_constraints=constraints)
        report['actual_compilation_passed'] = True
        report['effective_generation_config'] = agent.generate_content_config.model_dump(mode='json', exclude_none=True)
        requests = []
        async def respond(request):
            if request.url.host != '127.0.0.1' or request.url.path != '/v1/chat/completions':
                raise ValueError('Mock transport received unexpected endpoint')
            body = json.loads(request.content)
            if body.get('stream'):
                raise ValueError('This frozen diagnostic supports nonstreaming only')
            index = len(requests)
            if index >= 3:
                raise ValueError('Unexpected extra mock model request')
            requests.append(body)
            # No headers, credentials or real model content are captured.
            (root / f'http_request_{index:02d}.json').write_text(json.dumps(body, indent=2) + '\n', encoding='utf-8')
            name = ('load_skill', 'get_status')[index] if index < 2 else None
            args = {'skill_name': 'repair-evidence'} if index == 0 else {}
            message = {'role': 'assistant', 'content': 'Synthetic transport probe complete.' if name is None else None,
                'reasoning_content': f'SYNTHETIC_REASONING_SENTINEL_{index}'}
            if name is not None:
                message['tool_calls'] = [{'id': f'synthetic_call_{index}', 'type': 'function',
                    'function': {'name': name, 'arguments': json.dumps(args)}}]
            response = {'id': f'synthetic_completion_{index}', 'object': 'chat.completion',
                'created': 0, 'model': alias, 'choices': [{'index': 0, 'message': message,
                    'finish_reason': 'tool_calls' if name else 'stop'}],
                'usage': {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0}}
            return httpx.Response(200, json=response)
        client = AsyncOpenAI(api_key='EMPTY', base_url='http://127.0.0.1:8000/v1',
            max_retries=0, http_client=httpx.AsyncClient(transport=httpx.MockTransport(respond)))
        agent.model._additional_args['client'] = client
        service = InMemorySessionService()
        session = await service.create_session(app_name='synthetic_transport_probe', user_id='probe')
        runner = Runner(agent=agent, app_name='synthetic_transport_probe', session_service=service)
        events = []
        async for event in runner.run_async(user_id='probe', session_id=session.id,
                new_message=types.Content(role='user', parts=[types.Part(text=
                    'Synthetic transport-only check: load repair-evidence, get_status, then finish.')])):
            events.append(event.model_dump(mode='json', exclude_none=True))
        (root / 'synthetic_runner_events.json').write_text(json.dumps(events, indent=2) + '\n', encoding='utf-8')
        if len(requests) != 3:
            raise ValueError('Actual Runner did not make the three declared requests')
        report['actual_mocked_http_requests'] = len(requests)
        report['charged_tool_calls'] = context.tool_calls_used
        report['inference_server_process_absent'] = server.process is None
        model_path = Path(model_path)
        if not model_path.is_dir():
            raise ValueError('Exact attached 31B QAT revision 2 missing')
        report['model_path'] = str(model_path)
        report['tokenizer_files'] = {name: hashlib.sha256((model_path / name).read_bytes()).hexdigest()
            for name in ('chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json')
            if (model_path / name).is_file()}
        tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True, trust_remote_code=False)
        report['effective_chat_template_sha256'] = hashlib.sha256(tokenizer.chat_template.encode()).hexdigest()
        (root / 'attached_chat_template.jinja').write_text(tokenizer.chat_template, encoding='utf-8')
        postprocess, postprocessor_receipt = vllm_argument_postprocessor()
        report['installed_argument_postprocessor'] = postprocessor_receipt
        renders = []
        for index, body in enumerate(requests):
            messages = copy.deepcopy(body['messages'])
            postprocess(messages)
            tools = body.get('tools', [])
            names = [tool['function']['name'] for tool in tools]
            args_schema = next(tool['function']['parameters']['properties']['args']
                for tool in tools if tool['function']['name'] == 'run_skill_script')
            kwargs = dict(body.get('chat_template_kwargs', {}))
            if not isinstance(kwargs, dict):
                raise ValueError('Unexpected chat template kwargs')
            item = {'request_index': index, 'tool_names': names, 'run_skill_script_args_schema': args_schema,
                'actual_request_template_kwargs': kwargs,
                'reasoning_fields_on_wire': [{key: message[key] for key in ('reasoning', 'reasoning_content') if key in message}
                    for message in body['messages'] if message['role'] == 'assistant'], 'variants': []}
            for preserve in (None, True):
                settings = dict(kwargs)
                if preserve is not None:
                    settings['preserve_thinking'] = preserve
                variant = {'preserve_thinking_diagnostic_override': preserve}
                try:
                    rendered = tokenizer.apply_chat_template(messages, tools=tools, tokenize=False,
                        add_generation_prompt=True, **settings)
                    label = 'request_default' if preserve is None else 'diagnostic_preserve_true'
                    (root / f'rendered_{index:02d}_{label}.txt').write_text(rendered, encoding='utf-8')
                    variant.update(rendered=True, characters=len(rendered),
                        sha256=hashlib.sha256(rendered.encode()).hexdigest(),
                        all_declared_tool_names_present=all(name in rendered for name in names),
                        reasoning_sentinels_present=[f'SYNTHETIC_REASONING_SENTINEL_{i}' in rendered for i in range(index)])
                except Exception as error:
                    variant.update(rendered=False, error_type=type(error).__name__, error=str(error))
                item['variants'].append(variant)
            renders.append(item)
        report.update(status='completed', tokenizer_render_observations=renders,
            actual_request_default_all_rendered=all(item['variants'][0]['rendered'] for item in renders))
    except Exception as error:
        import traceback
        report.update(status='failed', error_type=type(error).__name__, error=str(error))
        (root / 'probe_exception.txt').write_text(traceback.format_exc(), encoding='utf-8')
    finally:
        if runner is not None:
            await runner.close()
        if client is not None:
            await client.close()
        if manager is not None:
            allowed = (root / 'synthetic_sandboxes').resolve()
            for paths in manager.sandboxes.values():
                if not paths['root'].resolve().is_relative_to(allowed):
                    raise ValueError('Unexpected synthetic cleanup root')
            manager.cleanup_all()
        save()
    return report


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    result = await probe_chat_transport(RUN, PLAN, PACKAGES, PLAN['model_path'], export_run)
    print('ACTUAL_CHAT_TRANSPORT_PROBE=' + json.dumps(result))
    export_run(RUN)
else:
    print('Definitions only; no SDK, model, tokenizer or grader executed locally')
